# Thinking about flow rates

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/04-flow-rates.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

## Risk and rates

The parameters applied to the flows in compartmental models represent the rate
of transition — for example the rate from source to destination in
[thinking about flows](./03-thinking-about-flows.ipynb). In epidemiology, field
data are often given as a **risk** over an illness episode or a period of time,
not as a per-unit-time rate. Relating those risks to the rates we put on flows
matters when we parameterise a model.


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS = {"index": "time", "value": "proportion"}


def state_frame(output: Any) -> pd.DataFrame:
    """Compartment output as a DataFrame with short state labels."""
    frame = output.to_pandas()
    frame.columns = [c.split("state=")[-1].split("_pop=")[0] for c in frame.columns]
    return frame



To get started, reuse the single-transition model from the previous chapter.


In [ ]:
def get_single_transition_model() -> tuple[Any, PropertyMap, Property]:
    """Two compartments linked by one per-capita transition flow."""
    state = Property("state", ("source", "destination"))
    pmap = PropertyMap.from_property(state)
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "transition",
            state["source"],
            state["destination"],
            Param("transition_rate"),
        )
    )
    return model.compile(), pmap, state


model_config = {"population": 1.0, "end_time": 20.0}
parameters = {"transition_rate": 0.1}

cm, pmap, state = get_single_transition_model()
y0 = np.zeros(pmap.size)
y0[pmap.select(state["source"])] = model_config["population"]
times = np.linspace(0.0, model_config["end_time"], int(model_config["end_time"]) + 1)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=1.0, save=plan, solver="dopri5")
outputs = state_frame(res["comp"])

risk_time = 10.0
dec_places = 3
risk_value = round(float(outputs["destination"].loc[risk_time]), dec_places)
print(f"The risk of reaching the destination compartment after ten time units is {risk_value}")
assert abs(risk_value - (1.0 - np.exp(-parameters["transition_rate"] * risk_time))) < 5e-3


As in the previous chapter, with a single outflow the destination size after
$t$ time units is

$$
\mathrm{risk} = 1 - e^{-\mathrm{rate}\, t}.
$$

Solving for rate,

$$
\mathrm{rate} = \frac{-\log(1 - \mathrm{risk})}{t}.
$$

If a cohort study said 63.2% of people reached the destination after ten time
units, this is how we would recover the rate to put on the flow.


In [ ]:
recalculated_rate = round(-np.log(1.0 - risk_value) / risk_time, dec_places)
print(
    f"To achieve a risk of {risk_value} after {round(risk_time)} time units, "
    f"we need a rate of {recalculated_rate}."
)
assert recalculated_rate == round(parameters["transition_rate"], dec_places)


## Competing flows

The same ideas extend when two transition flows compete for the same source.
The analyses below are deliberately simple — they are meant to build intuition
for how rates combine, not to model a real pathogen.

![](figures/04/source_two_dest_structure.svg)


In [ ]:
def get_competing_transition_model() -> tuple[Any, PropertyMap, Property]:
    """One source with two competing per-capita transitions."""
    state = Property("state", ("source", "destination_0", "destination_1"))
    pmap = PropertyMap.from_property(state)
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "transition_0",
            state["source"],
            state["destination_0"],
            Param("transition_0"),
        )
    )
    model.add_flow(
        TransitionFlow(
            "transition_1",
            state["source"],
            state["destination_1"],
            Param("transition_1"),
        )
    )
    return model.compile(), pmap, state


model_config = {"population": 1.0, "end_time": 200.0}
parameters = {"transition_0": 0.01, "transition_1": 0.02}

cm, pmap, state = get_competing_transition_model()
y0 = np.zeros(pmap.size)
y0[pmap.select(state["source"])] = model_config["population"]
times = np.linspace(0.0, model_config["end_time"], int(model_config["end_time"]) + 1)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=1.0, save=plan, solver="dopri5")
outputs = state_frame(res["comp"])
outputs.plot(labels=AXIS, title="Competing transition flows")


## Sojourn times

With no inward flows, the exit rate from a compartment is the sum of its
outflow rates. Here the mean time spent in `source` is the reciprocal of that
sum — the **sojourn time**.

If those rates are constant and there are no inward flows, the source size at
time $t$ is $e^{-\mathrm{outflows}\, t}$, and the mean sojourn is

$$
\int_0^\infty e^{-\mathrm{outflows}\, t}\, dt = \frac{1}{\mathrm{outflows}}.
$$


In [ ]:
outflow_sum = sum(parameters.values())
sojourn = 1.0 / outflow_sum
print(f"The average sojourn time for the source compartment is {round(sojourn, 2)} time units.")
assert round(sojourn, 2) == 33.33


We can check this numerically by weighting arrivals by the time they arrived
(a slight underestimate because the run is finite, not infinite).


In [ ]:
# People leaving source equal arrivals into the two destinations.
arrivals = outputs[["destination_0", "destination_1"]].diff().sum(axis=1)
weighted_arrival_times = arrivals * outputs.index
mean_arrival = float(weighted_arrival_times.sum())
print(f"The average time to arrive in a destination compartment is {mean_arrival}")
assert abs(mean_arrival - sojourn) < 1.0  # finite-horizon bias


## Median transition time

A sojourn time is a mean, not a median. Half the population has left `source`
when

$$
e^{-\mathrm{outflows}\, t} = \tfrac{1}{2}
\quad\Rightarrow\quad
t = \frac{-\log(\tfrac{1}{2})}{\mathrm{outflows}}.
$$


In [ ]:
median_t = -np.log(0.5) / outflow_sum
print(
    f"The time when half of the population have left the source is "
    f"{round(median_t, 3)} time units."
)
assert abs(median_t - np.log(2) / outflow_sum) < 1e-12


Again, check against the numerical trajectory.


In [ ]:
first_half = float(outputs["source"][outputs["source"].lt(0.5)].index[0])
print(
    f"The first time step with more than half the source compartment depleted is "
    f"{first_half} time units."
)
assert abs(first_half - median_t) < 1.0


## Risks from competing rates

Often we care about the risk of one outcome versus another for someone entering
a state — for example recovery versus infection fatality when both are outflows
from an infectious compartment. With constant competing rates and no other
exits, each outcome's risk is that flow's rate divided by the total outflow.


In [ ]:
risk0 = parameters["transition_0"] / outflow_sum
print(f"The risk of following transition_0 is {round(risk0 * 100)}%.")
assert round(risk0 * 100) == 33


The numeric solutions should settle on the same split.


In [ ]:
share = outputs["destination_0"] / outputs[["destination_0", "destination_1"]].sum(axis=1)
print(share.dropna().tail())
assert abs(float(share.iloc[-1]) - risk0) < 1e-3
share.plot(labels={"index": "time", "value": "share in destination_0"}, title="Outcome share")


These ideas matter when estimating parameters.

## Non-proportional hazards

Things are harder when the mix of outcomes depends on *when* we look. Suppose
30% recover and 70% die, recoveries average 5 days, and deaths average 10 days.
We cannot recover two exponential rates from those risks with the single-flow
formula above.

Options include representing the two processes with their own compartments so
the calculations separate, or accepting a single average sojourn and a fixed
outcome split. Constant exponential outflows cannot encode arbitrary
time-dependent relative hazards. Understanding the limitation is the point —
parameterisation has to respect what the flow structure can express.
